In [ ]:
# ============================================================
# 1) Environment setup
# ============================================================
import os, sys, re, time, random, shutil, zipfile, subprocess
from pathlib import Path
from datetime import datetime
# Kill leftover DDP workers from an interrupted run (they keep holding GPU memory)
try:
    subprocess.run(["pkill", "-9", "-f", "[U]ultralytics/DDP"], check=False)
    time.sleep(3)
except FileNotFoundError:
    pass
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["YOLO_CONFIG_DIR"] = "/tmp/Ultralytics"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

IN_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path(".")
print("IN_KAGGLE =", IN_KAGGLE)
print("WORK_DIR  =", WORK_DIR)
print("Python    =", sys.version)

def run_pip_install(packages):
    if isinstance(packages, str):
        packages = [packages]
    cmd = [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--disable-pip-version-check", *packages]
    print("Running:", " ".join(cmd))
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print("pip return code:", p.returncode)
    if p.stdout: print("----- pip stdout -----\n", p.stdout[-6000:])
    if p.stderr: print("----- pip stderr -----\n", p.stderr[-6000:])
    if p.returncode != 0:
        raise RuntimeError("pip install failed. Enable Kaggle Internet, restart session, then run again.")

try:
    import ultralytics
    print("ultralytics:", ultralytics.__version__)
except Exception as e:
    print("Ultralytics missing:", repr(e))
    run_pip_install(["ultralytics"])
    import ultralytics
    print("ultralytics:", ultralytics.__version__)

import yaml
import pandas as pd
import numpy as np
import cv2
from ultralytics import YOLO

try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            prop = torch.cuda.get_device_properties(i)
            print(i, torch.cuda.get_device_name(i), round(prop.total_memory / 1024**3, 2), "GB")
except Exception as e:
    print("Torch info error:", repr(e))

In [ ]:
# ============================================================
# 2) YOLOv9e fine-tuning configuration
# ============================================================
# This run fine-tunes COCO-pretrained YOLOv9e (yolov9e.pt) on the plate
# dataset and exports the final detector as v63.pt.
# It no longer continues from the old v62.pt checkpoint.
# ============================================================

VERSION = "V63_YOLOV9E_FINETUNE_26E"

DATASET_FOLDER_NAME = "v58_merged_dataset_real"
PREFERRED_DATASET_ROOTS = [
    Path("/kaggle/input/datasets/thnhtrung5/vn-licence/dataset/v58_merged_dataset_real/v58_merged_dataset_real"),
]

def find_dataset_root():
    for candidate in PREFERRED_DATASET_ROOTS:
        if (candidate / "train" / "images").exists() and (candidate / "valid" / "images").exists():
            return candidate

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for candidate in kaggle_input.rglob(DATASET_FOLDER_NAME):
            if (candidate / "train" / "images").exists() and (candidate / "valid" / "images").exists():
                return candidate

    local_candidate = Path(DATASET_FOLDER_NAME)
    if (local_candidate / "train" / "images").exists() and (local_candidate / "valid" / "images").exists():
        return local_candidate.resolve()

    return PREFERRED_DATASET_ROOTS[0]

DATASET_ROOT = find_dataset_root()
INPUT_ROOT = DATASET_ROOT.parent

# Base weights: COCO-pretrained YOLOv9e.
# Ultralytics downloads "yolov9e.pt" automatically (Kaggle Internet must be ON).
# If you attach a Kaggle dataset that contains yolov9e.pt, that copy is used instead.
BASE_MODEL = "Wiliv4.pt"
FINAL_MODEL_NAME = "Wiliv5.pt"  # final detector file exported to WORK_DIR

def find_base_weights():
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.rglob(BASE_MODEL):
            if p.is_file():
                return str(p)
    return BASE_MODEL

BASE_WEIGHTS = find_base_weights()

EPOCHS = 3  # epochs for this YOLOv9e fine-tuning run
IMG_SIZE = 768

# YOLOv9e is much heavier than a small/medium YOLO. Keep ~8 images per GPU at
# imgsz 768 to avoid CUDA OOM on a 16 GB T4/P100:
#   2 GPUs -> batch 16, 1 GPU (or CPU) -> batch 8.
# If you still get 'CUDA out of memory', lower PER_GPU_BATCH to 4.
PER_GPU_BATCH = 4
try:
    import torch
    N_GPU = torch.cuda.device_count()
except Exception:
    N_GPU = 0
BATCH = PER_GPU_BATCH * min(max(N_GPU, 1), 2)  # pick_device() uses at most 2 GPUs
WORKERS = 2
AMP = True
DEVICE = "auto"

# Lower LR is safer when fine-tuning from pretrained weights.
LR0 = 0.002
LRF = 0.01
WARMUP_EPOCHS = 2.0
PATIENCE = 8

MOSAIC = 0.35
CLOSE_MOSAIC = 1
COS_LR = True
SEED = 22
SAVE_PERIOD = 5 

PROJECT = WORK_DIR / "runs_vnlp_public_v63_yolov9e"
RUN_NAME = "v63_yolov9e_finetune_25e"
ZIP_NAME = "vnlp_public_v63_yolov9e.zip"
DATA_YAML = WORK_DIR / "v63_yolov9e_dataset.yaml"

RUN_YOLO_PREVIEW = True
RUN_OCR_PREVIEW = True
PREVIEW_CONF = 0.15
PREVIEW_MAX_IMAGES = 32
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("VERSION        =", VERSION)
print("DATASET_ROOT   =", DATASET_ROOT)
print("BASE_WEIGHTS   =", BASE_WEIGHTS)
print("EPOCHS         =", EPOCHS)
print("BATCH          =", BATCH, f"({N_GPU} GPU)")
print("MODE           = fine-tune YOLOv9e ->", FINAL_MODEL_NAME)

In [ ]:
# ============================================================
# 3) Verify dataset and write data.yaml
# ============================================================
def count_images(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in IMG_EXTS)

def count_labels(folder: Path) -> int:
    if not folder.exists():
        return 0
    return len(list(folder.rglob("*.txt")))

for p in [
    DATASET_ROOT / "train" / "images",
    DATASET_ROOT / "train" / "labels",
    DATASET_ROOT / "valid" / "images",
    DATASET_ROOT / "valid" / "labels",
    DATASET_ROOT / "test" / "images",
    DATASET_ROOT / "test" / "labels",
]:
    print(p, "exists=", p.exists())

if not (DATASET_ROOT / "train" / "images").exists():
    raise FileNotFoundError("Missing train/images")
if not (DATASET_ROOT / "valid" / "images").exists():
    raise FileNotFoundError("Missing valid/images")

rows = []
for split in ["train", "valid", "test"]:
    rows.append({
        "split": split,
        "images": count_images(DATASET_ROOT / split / "images"),
        "labels": count_labels(DATASET_ROOT / split / "labels"),
        "image_dir": str(DATASET_ROOT / split / "images"),
        "label_dir": str(DATASET_ROOT / split / "labels"),
    })

stats_df = pd.DataFrame(rows)
display(stats_df)

test_images = count_images(DATASET_ROOT / "test" / "images")
data_yaml = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images" if test_images > 0 else "valid/images",
    "names": {0: "license_plate"},
    "nc": 1,
}
DATA_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

In [ ]:
# ============================================================
# 4) Label sanity check
# ============================================================
# YOLO detect labels must have exactly 5 columns:
# class x_center y_center width height
# ============================================================

def check_label_files(label_root: Path, max_bad_print=20):
    bad_files, total_files, total_boxes = [], 0, 0
    for txt_path in label_root.rglob("*.txt"):
        total_files += 1
        text = txt_path.read_text(encoding="utf-8", errors="ignore")
        if "\\n" in text:
            bad_files.append(txt_path)
            if len(bad_files) <= max_bad_print:
                print("BAD literal backslash-n:", txt_path)
            continue
        for line in text.splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 5:
                bad_files.append(txt_path)
                if len(bad_files) <= max_bad_print:
                    print("BAD column count:", txt_path, "line=", repr(line))
                break
            try:
                _ = float(parts[0])
                vals = [float(x) for x in parts[1:]]
                if not all(np.isfinite(vals)):
                    raise ValueError("non-finite")
                total_boxes += 1
            except Exception:
                bad_files.append(txt_path)
                if len(bad_files) <= max_bad_print:
                    print("BAD float parse:", txt_path, "line=", repr(line))
                break
    print("Checked label files:", total_files)
    print("Checked boxes:", total_boxes)
    print("Bad label files:", len(bad_files))
    if bad_files:
        raise RuntimeError("Invalid YOLO label formatting found.")

for split in ["train", "valid", "test"]:
    label_dir = DATASET_ROOT / split / "labels"
    if label_dir.exists():
        print(f"Checking {split} labels:", label_dir)
        check_label_files(label_dir)

In [ ]:
# # ============================================================
# # 6) Build YOLOv9e model
# # ============================================================
# # Loads COCO-pretrained YOLOv9e weights. The final detection layers are
# # re-initialised for the 1-class dataset (license_plate) when training starts.
# # ============================================================

# print("Loading base weights:", BASE_WEIGHTS)
# try:
#     model = YOLO(BASE_WEIGHTS)
# except Exception as e:
#     raise RuntimeError(
#         f"Cannot load {BASE_WEIGHTS}. If Kaggle Internet is OFF, turn it ON in the notebook "
#         "settings (Ultralytics downloads yolov9e.pt from GitHub), or attach a dataset "
#         "that contains yolov9e.pt."
#     ) from e
# model.info(verbose=False)
# print("Base model loaded. A new fine-tuning run will start from these weights.")


In [ ]:
# # ============================================================
# # 7) Fine-tune YOLOv9e
# # ============================================================
# start_time = time.time()

# train_result = model.train(
#     data=str(DATA_YAML),
#     epochs=EPOCHS,
#     imgsz=IMG_SIZE,
#     batch=BATCH,
#     workers=WORKERS,
#     amp=AMP,

#     project=str(PROJECT),
#     name=RUN_NAME,
#     exist_ok=True,
#     save=True,
#     save_period=SAVE_PERIOD,
#     plots=True,

#     optimizer="auto",
#     lr0=LR0,
#     lrf=LRF,
#     warmup_epochs=WARMUP_EPOCHS,
#     patience=PATIENCE,
#     cos_lr=COS_LR,
#     mosaic=MOSAIC,
#     close_mosaic=CLOSE_MOSAIC,
#     seed=SEED,
#     cache=False,
# )

# RUN_DIR = PROJECT / RUN_NAME
# elapsed_hours = (time.time() - start_time) / 3600

# print("Fine-tuning finished.")
# print("Started from base weights:", BASE_WEIGHTS)
# print("Elapsed hours:", round(elapsed_hours, 3))
# print("RUN_DIR:", RUN_DIR)

In [ ]:
# ============================================================
# 7b) TRAIN TIẾP từ checkpoint (thay thế cell "6) Build model" + "7) Fine-tune")
# ============================================================
import shutil, torch
from pathlib import Path
from ultralytics import YOLO

# ---------- Đường dẫn / cấu hình (chỉnh ở đây) ----------
# Tự tìm Wiliv1.pt trong Kaggle Input, không phụ thuộc tên dataset / chữ hoa-thường
CKPT_SRC     = next(p for p in Path("/kaggle/input").rglob("*.pt") if p.name.lower() == "wiliv4.pt")
CKPT_LOCAL   = WORK_DIR / "resume_last.pt"   # bản copy ghi được (/kaggle/input là read-only)
MODE         = "new_finetune"                # "new_finetune" | "resume" | "extend"
EXTRA_EPOCHS = 9        # new_finetune: số epoch train thêm (mỗi epoch ~2 giờ)
TOTAL_EPOCHS = 10       # extend: tổng số epoch của cả run (không dùng ở new_finetune)
NEW_LR0      = 0.0002   # new_finetune: learning rate khởi đầu
DEVICE_TRAIN = [0, 1] if torch.cuda.device_count() >= 2 else 0
RUN_NAME_CONT = RUN_NAME + "_cont"           # thư mục run mới của new_finetune
# ---------------------------------------------------------

assert MODE in ("new_finetune", "resume", "extend"), f"MODE không hợp lệ: {MODE}"
print("Checkpoint nguồn:", CKPT_SRC)

shutil.copy2(CKPT_SRC, CKPT_LOCAL)
ckpt = torch.load(CKPT_LOCAL, map_location="cpu", weights_only=False)

if ckpt.get("epoch", -1) == -1 or ckpt.get("optimizer") is None:
    print("Checkpoint đã strip (train xong / không còn optimizer) -> không resume được, dùng new_finetune")
    MODE = "new_finetune"
else:
    print(f"Đã xong {ckpt['epoch'] + 1} epoch | run gốc đặt tổng {ckpt['train_args']['epochs']} epoch")

if MODE == "resume":
    YOLO(str(CKPT_LOCAL)).train(resume=True, device=DEVICE_TRAIN)

elif MODE == "extend":
    ckpt["train_args"]["epochs"] = TOTAL_EPOCHS
    torch.save(ckpt, CKPT_LOCAL)
    YOLO(str(CKPT_LOCAL)).train(resume=True, device=DEVICE_TRAIN)

else:  # new_finetune
    YOLO(str(CKPT_LOCAL)).train(
        data=str(DATA_YAML), epochs=EXTRA_EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
        workers=WORKERS, amp=AMP, device=DEVICE_TRAIN,
        project=str(PROJECT), name=RUN_NAME_CONT, exist_ok=False,
        optimizer="AdamW", lr0=NEW_LR0, lrf=LRF, warmup_epochs=0.5, cos_lr=COS_LR,
        patience=PATIENCE, mosaic=MOSAIC, close_mosaic=CLOSE_MOSAIC,
        seed=SEED, cache=False, save_period=SAVE_PERIOD, plots=True,
    )

In [ ]:
# ============================================================
# 8) METRICS CELL: Precision, Recall, mAP50, mAP50-95
# ============================================================
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path
from IPython.display import display
from ultralytics import YOLO

# ---------- Cấu hình (chỉnh ở đây) ----------
PROJECT_DIR = Path(PROJECT)
RUN_PREFIX  = RUN_NAME + "_cont"                  # prefix thư mục run của new_finetune
EXTRA_ROOT  = Path("/kaggle/working/runs")        # nơi Ultralytics có thể lưu nếu PROJECT là đường dẫn tương đối
METRICS_OUT = WORK_DIR / "v63_yolov9e_metrics_summary.csv"
RUN_VAL     = False                               # True: validate lại best.pt trên val set (chậm hơn nhưng chắc chắn nhất)
# --------------------------------------------

# 1) Tìm tất cả run khớp prefix, sắp xếp mới nhất lên đầu
roots = [PROJECT_DIR] + ([EXTRA_ROOT] if EXTRA_ROOT.exists() else [])
csvs = {}
for root in roots:
    for p in root.rglob("results.csv"):
        if p.parent.name.startswith(RUN_PREFIX):
            csvs[p.resolve()] = p
candidates = sorted(csvs.values(), key=lambda p: p.stat().st_mtime, reverse=True)

if not candidates:
    raise FileNotFoundError(f"Không thấy results.csv nào có prefix '{RUN_PREFIX}' trong {[str(r) for r in roots]}")

print("Các run tìm thấy (mới nhất trước):")
for p in candidates:
    n_rows = len(pd.read_csv(p))
    mtime = datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
    print(f"  {p.parent} | {n_rows} epoch | sửa lần cuối: {mtime}")

results_csv = candidates[0]                       # run mới nhất
RUN_DIR = results_csv.parent
best_pt = RUN_DIR / "weights" / "best.pt"
last_pt = RUN_DIR / "weights" / "last.pt"
print("\n>>> Đang dùng RUN_DIR:", RUN_DIR)
print("best.pt:", best_pt, "| exists:", best_pt.exists())
print("last.pt:", last_pt, "| exists:", last_pt.exists())

# 2) Đọc results.csv
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
display(df.tail(10))

metric_cols = {
    "Precision": "metrics/precision(B)",
    "Recall":    "metrics/recall(B)",
    "mAP50":     "metrics/mAP50(B)",
    "mAP50-95":  "metrics/mAP50-95(B)",
}

# best.pt của Ultralytics được chọn theo fitness = 0.1*mAP50 + 0.9*mAP50-95
df["fitness"] = 0.1 * df[metric_cols["mAP50"]] + 0.9 * df[metric_cols["mAP50-95"]]

def row_to_dict(label, row):
    d = {"Selection": label, "Epoch": int(row["epoch"]) if "epoch" in row.index else -1}
    for name, col in metric_cols.items():
        d[name] = float(row[col]) if col in row.index else np.nan
    return d

last_metrics = row_to_dict("Last epoch", df.iloc[-1])
best_metrics = row_to_dict("Best (khớp best.pt)", df.loc[df["fitness"].idxmax()])

metrics_table = pd.DataFrame([last_metrics, best_metrics])
display(metrics_table)

# 3) (Tùy chọn) validate trực tiếp best.pt để đối chiếu
if RUN_VAL:
    val = YOLO(str(best_pt)).val(data=str(DATA_YAML), imgsz=IMG_SIZE, batch=BATCH,
                                 device=0, plots=False)
    val_row = {"Selection": "val() on best.pt", "Epoch": -1,
               "Precision": float(val.box.mp), "Recall": float(val.box.mr),
               "mAP50": float(val.box.map50), "mAP50-95": float(val.box.map)}
    metrics_table = pd.concat([metrics_table, pd.DataFrame([val_row])], ignore_index=True)
    display(metrics_table)

metrics_table.to_csv(METRICS_OUT, index=False, encoding="utf-8-sig")
print("Saved metrics summary:", METRICS_OUT)

In [ ]:
# # ============================================================
# # 9) Validate best.pt
# # ============================================================

# if best_pt.exists():
#     print("Validating best.pt...")
#     val_model = YOLO(str(best_pt))
#     val_metrics = val_model.val(
#         data=str(DATA_YAML),
#         imgsz=IMG_SIZE,
#         batch=BATCH,
#     )
#     print(val_metrics)
# else:
#     print("Validation skipped: best.pt missing.")

In [ ]:
# ============================================================
# 8) XUẤT FILE ĐỂ TẢI VỀ
# ============================================================
import shutil
from pathlib import Path
from IPython.display import FileLink, display
from ultralytics import YOLO

# ---------- Đường dẫn / cấu hình (chỉnh ở đây) ----------
RUN_DIR     = Path(PROJECT) / RUN_NAME_CONT     # thư mục run vừa train
EXPORT_DIR  = Path("/kaggle/working/export")    # nơi gom file để tải
OUT_NAME    = "Wiliv5"                          # tên file xuất ra
EXPORT_ONNX = False                             # True nếu cần thêm file .onnx
MAKE_ZIP    = True                              # True nếu muốn nén cả run (plots, results.csv)
# ---------------------------------------------------------

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

best = RUN_DIR / "weights" / "best.pt"
last = RUN_DIR / "weights" / "last.pt"
assert best.exists(), f"Không thấy {best} -> kiểm tra lại PROJECT / RUN_NAME_CONT"

shutil.copy2(best, EXPORT_DIR / f"{OUT_NAME}_best.pt")
if last.exists():
    shutil.copy2(last, EXPORT_DIR / f"{OUT_NAME}_last.pt")

if EXPORT_ONNX:
    onnx_path = YOLO(str(best)).export(format="onnx", imgsz=IMG_SIZE, simplify=True)
    shutil.copy2(onnx_path, EXPORT_DIR / f"{OUT_NAME}_best.onnx")

if MAKE_ZIP:
    shutil.make_archive(str(EXPORT_DIR / f"{OUT_NAME}_run"), "zip", root_dir=RUN_DIR)

# In danh sách + link tải
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"{f.name:30s} {f.stat().st_size / 1e6:8.1f} MB")
    display(FileLink(str(f.relative_to("/kaggle/working"))))